# 1. Install necessary Libraries

In this part, we'll begin by installing necessary libraries needed for running our computer vision training and testing scripts

In [1]:
%pip install torch torchvision torchaudio
%pip install transformers scikit-learn pillow pandas numpy huggingface_hub ipywidgets opencv-python

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 30.7 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 22.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.9/679.9 kB 11.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 27.1 MB/s  0:00:00
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [torchvision] [torchvision]
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# 2. Huggingface login via UI
This part is optional as you're still able to access huggingface API without authentication, just that there's stricter rate limiting. If you have not faced any rate limiting issues can comment out or delete this cell

In [2]:
from huggingface_hub import notebook_login
notebook_login()

# 3. CLAHE Data Preprocessing function
In this part, we'll write Contrast Limited Adaptive Histogram Equalization (CLAHE) preprocessing function logic. Since this data preprocessing is added to all dataset (training/val/test) and its not part of data augmentation, we'll separate them for now

In [3]:
import cv2
import numpy as np
from PIL import Image

def apply_clahe(image):
    image = np.array(image)

    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(l)

    merged = cv2.merge((cl, a, b))
    enhanced = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

    return Image.fromarray(enhanced)

# 4. Custom Dataset Loader
Since we're using a custom dataset, we will need to write a custom dataset loader to pass our image data to the model


In [4]:
from torch.utils.data import Dataset
from PIL import Image
import os

class MedicalDatasetLoader(Dataset):
    def __init__(self, df, img_dir, transform=None, use_clahe=True):
        self.data = df # Differentiate between train_df, val_df and test_df
        self.img_dir = img_dir # Image Directory Path
        self.transform = transform # Data augmentation
        self.use_clahe = use_clahe # Apply CLAHE data preprocessing

    def __len__(self):
        return len(self.data) # Calculate the number of rows of the dataset

    def __getitem__(self, idx):
        img_name = self.data.iloc[idx]['Image'] # Get the image name from the csv header. Used for path later
        label = int(self.data.iloc[idx]['Label']) # Get the Label value from csv header

        img_path = os.path.join(self.img_dir, img_name) # Combine the root image dir with the image name to get the full individual image path
        image = Image.open(img_path).convert("RGB")

        if self.use_clahe: # Implement CLAHE for training set, disable for val and testing set
            image = apply_clahe(image)

        if self.transform: # Implement Data Augmentation
            image = self.transform(image)

        return image, label

# 5. Setup MPS Device
Here in this part, we check if PyTorch detects our GPU MPS device on macOS

In [5]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu") # Check if MPS is detected by pytorch, fallback to CPU otherwise
print("Using device:", device)

Using device: mps


# 6. Setup Model
In this part, we will load the SwinV2 (Swin Transformer V2) model from HuggingFace. The model is pretrained on ImageNet-1K at 256x256 resolution but we use 224x224 for cross-model consistency.

In [6]:
from transformers import Swinv2ForImageClassification, AutoImageProcessor

# Load Microsoft SwinV2 model
model = Swinv2ForImageClassification.from_pretrained(
    'microsoft/swinv2-base-patch4-window8-256',
    num_labels=5,
    ignore_mismatched_sizes=True
).to(device)

processor = AutoImageProcessor.from_pretrained('microsoft/swinv2-base-patch4-window8-256')

config.json: 0.00B [00:00, ?B/s]

You passed `num_labels=5` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin:   0%|          | 0.00/352M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Swinv2ForImageClassification LOAD REPORT from: microsoft/swinv2-base-patch4-window8-256
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([5])            
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 1024]) vs model:torch.Size([5, 1024])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


preprocessor_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/352M [00:00<?, ?B/s]

# 7. Data Preprocessing & K-Fold Setup
In this part, we add data augmentation to the training transforms and set up Stratified K-Fold cross-validation. We hold out 20% as a fixed test set, then use 5-fold StratifiedKFold on the remaining 80%. DataLoaders are created per-fold inside the training loop.

In [7]:
from torchvision import transforms
from sklearn.model_selection import train_test_split, StratifiedKFold
from torch.utils.data import DataLoader, WeightedRandomSampler
import pandas as pd

# Data augmentation for training (applied after CLAHE)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

# No augmentation for validation and testing
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

df = pd.read_csv('dataset/labels.csv')

# Step 1: Hold out 20% as fixed test set (never touched during K-Fold)
train_val_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['Label'],
    random_state=42
)

# Step 2: Setup 5-Fold Stratified K-Fold on the remaining 80%
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Create test dataset and loader (fixed across all folds)
test_dataset = MedicalDatasetLoader(test_df, "dataset/image/", val_transform, use_clahe=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Total samples: {len(df)}")
print(f"Train+Val pool: {len(train_val_df)} (will be split into {N_FOLDS} folds)")
print(f"Test (held out): {len(test_df)}")

Total samples: 4939
Train+Val pool: 3951 (will be split into 5 folds)
Test (held out): 988


# 8. Hyperparameters tuning
We set up our class weights, epochs number, early stopping mechanism, optimizer and learning rate scheduler to optimize our model performance and provide better generalization through reducing overfitting

In [8]:
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.utils.class_weight import compute_class_weight

class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.alpha = alpha  # Per-class weights tensor on device
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        # Apply label smoothing via cross_entropy
        ce_loss = F.cross_entropy(
            logits, targets,
            weight=self.alpha,
            label_smoothing=self.label_smoothing,
            reduction='none'
        )
        # Get predicted probabilities for focal modulation
        probs = F.softmax(logits, dim=1)
        p_t = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        # Apply focal modulation: (1 - p_t)^gamma
        focal_weight = (1 - p_t) ** self.gamma
        loss = focal_weight * ce_loss
        return loss.mean()

# Shared hyperparameter constants
NUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 5
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 16

# 9. Experiment Logging Class
In this part, we'll create a log class tha can help us to logs our hyperparameter lists as well as result metrics.

In [9]:
import json
from datetime import datetime
import os

class ExperimentTracker:
    def __init__(self, base_dir="experiments"):
        self.base_dir = base_dir
        os.makedirs(base_dir, exist_ok=True)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.exp_dir = os.path.join(base_dir, f"exp_{timestamp}")
        os.makedirs(self.exp_dir)

        self.epoch_metrics = []
        self.fold_metrics = []
        self.final_metrics = {}
        self.config = {}

    # ---------------- CONFIG ----------------
    def log_config(self, config):
        self.config = config
        with open(os.path.join(self.exp_dir, "config.json"), "w") as f:
            json.dump(config, f, indent=4)

    # ---------------- PER EPOCH ----------------
    def log_epoch(self, fold, epoch, train_loss, val_loss, train_acc, val_acc):
        self.epoch_metrics.append({
            "fold": fold,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_acc": train_acc,
            "val_acc": val_acc
        })

    # ---------------- PER FOLD ----------------
    def log_fold_metrics(self, fold, acc, prec, rec, f1, roc_auc):
        self.fold_metrics.append({
            "fold": fold,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1_score": f1,
            "roc_auc_score": roc_auc
        })

    # ---------------- AGGREGATED K-FOLD ----------------
    def log_aggregated_metrics(self):
        if not self.fold_metrics:
            return {}
        metrics_keys = ["accuracy", "precision", "recall", "f1_score", "roc_auc_score"]
        aggregated = {}
        for key in metrics_keys:
            values = [fm[key] for fm in self.fold_metrics]
            aggregated[key] = {
                "mean": float(np.mean(values)),
                "std": float(np.std(values)),
                "per_fold": values
            }
        self.final_metrics["kfold_aggregated"] = aggregated
        return aggregated

    # ---------------- FINAL METRICS ----------------
    def log_final_metrics(self, split, acc, prec, rec, f1, roc_auc, cm):
        self.final_metrics[split] = {
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1_score": f1,
            "roc_auc_score": roc_auc,
            "confusion_matrix": cm.tolist()
        }

    # ---------------- SAVE ----------------
    def save_all(self):
        with open(os.path.join(self.exp_dir, "metrics.json"), "w") as f:
            json.dump({
                "config": self.config,
                "epoch_metrics": self.epoch_metrics,
                "fold_metrics": self.fold_metrics,
                "final_metrics": self.final_metrics
            }, f, indent=4)

    def save_model(self, model, name="best_model.pth"):
        torch.save(model.state_dict(), os.path.join(self.exp_dir, name))

# 10. Training and Validation Script
In this part, we train and test our model and finally save the final weights of the model

In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

CONFIG = {
    "model": "microsoft/swinv2-base-patch4-window8-256",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "scheduler": "CosineAnnealingLR",
    "loss": "FocalLoss",
    "focal_gamma": 2.0,
    "label_smoothing": 0.1,
    "k_folds": N_FOLDS,
    "augmentation": "HFlip+VFlip+Rotation15+ColorJitter",
    "sampler": "WeightedRandomSampler"
}

tracker = ExperimentTracker()
tracker.log_config(CONFIG)

# Track best model across all folds
global_best_val_loss = float('inf')
best_fold = -1

for fold, (train_idx, val_idx) in enumerate(skf.split(train_val_df, train_val_df['Label'])):
    print(f"\n{'='*60}")
    print(f"FOLD {fold + 1}/{N_FOLDS}")
    print(f"{'='*60}")

    # --- Split data for this fold ---
    fold_train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
    fold_val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

    # --- Create datasets ---
    fold_train_dataset = MedicalDatasetLoader(fold_train_df, "dataset/image/", train_transform, use_clahe=True)
    fold_val_dataset = MedicalDatasetLoader(fold_val_df, "dataset/image/", val_transform, use_clahe=False)

    # --- WeightedRandomSampler for this fold's training data ---
    fold_labels = fold_train_df['Label'].values
    class_counts = np.bincount(fold_labels)
    sample_weights = 1.0 / class_counts[fold_labels]
    sample_weights = torch.DoubleTensor(sample_weights)
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    # --- DataLoaders ---
    fold_train_loader = DataLoader(fold_train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
    fold_val_loader = DataLoader(fold_val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # --- Re-initialize model fresh for each fold ---
    model = Swinv2ForImageClassification.from_pretrained(
        'microsoft/swinv2-base-patch4-window8-256',
        num_labels=5,
        ignore_mismatched_sizes=True
    ).to(device)

    # --- Compute class weights for this fold ---
    class_weights = compute_class_weight('balanced', classes=np.unique(fold_train_df['Label']), y=fold_train_df['Label'])
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

    # --- Loss, optimizer, scheduler ---
    criterion = FocalLoss(alpha=class_weights, gamma=2.0, label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

    # --- Early stopping state ---
    best_val_loss = float('inf')
    epochs_no_improvement = 0

    for epoch in range(NUM_EPOCHS):
        # ---------------- TRAINING ----------------
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        for inputs, labels in fold_train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(pixel_values=inputs)
            loss = criterion(outputs.logits, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.logits, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        # ---------------- VALIDATION ----------------
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        all_preds, all_labels, all_probs = [], [], []

        with torch.no_grad():
            for inputs, labels in fold_val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(pixel_values=inputs)
                loss = criterion(outputs.logits, labels)

                probs = F.softmax(outputs.logits, dim=1)

                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.logits, 1)

                val_total += inputs.size(0)
                val_correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs.detach().cpu().numpy())

        # ---------------- EPOCH METRICS ----------------
        epoch_train_loss = train_loss / train_total
        epoch_train_acc = train_correct / train_total
        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total

        scheduler.step()

        tracker.log_epoch(fold + 1, epoch, epoch_train_loss, epoch_val_loss, epoch_train_acc, epoch_val_acc)

        # --- Early stopping ---
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            epochs_no_improvement = 0
            # Save if this is the best model across ALL folds
            if epoch_val_loss < global_best_val_loss:
                global_best_val_loss = epoch_val_loss
                best_fold = fold + 1
                tracker.save_model(model)
        else:
            epochs_no_improvement += 1
            if epochs_no_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"  Early stopping at epoch {epoch + 1}")
                break

        print(f"  Epoch [{epoch+1}/{NUM_EPOCHS}] Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")

    # --- End-of-fold validation metrics ---
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='macro')
    try:
        roc_auc = roc_auc_score(np.array(all_labels), np.array(all_probs), multi_class='ovr')
    except ValueError:
        roc_auc = 0.0

    tracker.log_fold_metrics(fold + 1, acc, prec, rec, f1, roc_auc)
    print(f"\n  Fold {fold+1} Val — Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | AUC: {roc_auc:.4f}")

# --- Aggregated K-Fold results ---
aggregated = tracker.log_aggregated_metrics()
print(f"\n{'='*60}")
print(f"K-FOLD AGGREGATED RESULTS (best model from fold {best_fold})")
print(f"{'='*60}")
for metric, vals in aggregated.items():
    print(f"  {metric}: {vals['mean']*100:.2f}% (+/- {vals['std']*100:.2f}%)")

tracker.save_all()


FOLD 1/5


You passed `num_labels=5` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Swinv2ForImageClassification LOAD REPORT from: microsoft/swinv2-base-patch4-window8-256
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([5])            
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 1024]) vs model:torch.Size([5, 1024])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  Epoch [1/20] Train Loss: 0.8616 | Train Acc: 0.5415 | Val Loss: 0.5615 | Val Acc: 0.6713
  Epoch [2/20] Train Loss: 0.5127 | Train Acc: 0.7142 | Val Loss: 0.6730 | Val Acc: 0.5234
  Epoch [3/20] Train Loss: 0.3794 | Train Acc: 0.7877 | Val Loss: 0.5387 | Val Acc: 0.7560
  Epoch [4/20] Train Loss: 0.3064 | Train Acc: 0.8215 | Val Loss: 0.7056 | Val Acc: 0.7118
  Epoch [5/20] Train Loss: 0.2230 | Train Acc: 0.8649 | Val Loss: 0.8665 | Val Acc: 0.7358
  Epoch [6/20] Train Loss: 0.1921 | Train Acc: 0.9051 | Val Loss: 0.7912 | Val Acc: 0.7054
  Epoch [7/20] Train Loss: 0.1565 | Train Acc: 0.9161 | Val Loss: 1.1419 | Val Acc: 0.6827
  Early stopping at epoch 8

  Fold 1 Val — Acc: 0.6448 | Prec: 0.6315 | Rec: 0.5692 | F1: 0.5537 | AUC: 0.8375

FOLD 2/5


...

  Epoch [1/20] Train Loss: 0.9253 | Train Acc: 0.4802 | Val Loss: 0.5380 | Val Acc: 0.5911
  Epoch [2/20] Train Loss: 0.5425 | Train Acc: 0.7014 | Val Loss: 0.5888 | Val Acc: 0.7152
  Epoch [3/20] Train Loss: 0.3996 | Train Acc: 0.7732 | Val Loss: 0.5482 | Val Acc: 0.6392
  Epoch [4/20] Train Loss: 0.2763 | Train Acc: 0.8333 | Val Loss: 0.7122 | Val Acc: 0.7848
  Epoch [5/20] Train Loss: 0.2607 | Train Acc: 0.8671 | Val Loss: 0.7805 | Val Acc: 0.6835
  Early stopping at epoch 6

  Fold 4 Val — Acc: 0.6848 | Prec: 0.6520 | Rec: 0.6374 | F1: 0.6131 | AUC: 0.9093

FOLD 5/5


You passed `num_labels=5` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Swinv2ForImageClassification LOAD REPORT from: microsoft/swinv2-base-patch4-window8-256
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([5])            
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 1024]) vs model:torch.Size([5, 1024])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  Epoch [1/20] Train Loss: 0.9364 | Train Acc: 0.4837 | Val Loss: 0.5660 | Val Acc: 0.6215
  Epoch [2/20] Train Loss: 0.6026 | Train Acc: 0.6691 | Val Loss: 0.7624 | Val Acc: 0.5861
  Epoch [3/20] Train Loss: 0.4796 | Train Acc: 0.7400 | Val Loss: 0.6151 | Val Acc: 0.5127
  Epoch [4/20] Train Loss: 0.3569 | Train Acc: 0.8045 | Val Loss: 0.5660 | Val Acc: 0.6608
  Epoch [5/20] Train Loss: 0.2665 | Train Acc: 0.8519 | Val Loss: 0.7775 | Val Acc: 0.6633
  Epoch [6/20] Train Loss: 0.2101 | Train Acc: 0.8820 | Val Loss: 0.6608 | Val Acc: 0.7076
  Epoch [7/20] Train Loss: 0.1839 | Train Acc: 0.9038 | Val Loss: 0.7874 | Val Acc: 0.7165
  Epoch [8/20] Train Loss: 0.1416 | Train Acc: 0.9212 | Val Loss: 0.8469 | Val Acc: 0.7633
  Early stopping at epoch 9

  Fold 5 Val — Acc: 0.7658 | Prec: 0.6771 | Rec: 0.6215 | F1: 0.6099 | AUC: 0.8858

K-FOLD AGGREGATED RESULTS (best model from fold 4)
  accuracy: 71.98% (+/- 4.83%)
  precision: 66.04% (+/- 1.89%)
  recall: 61.71% (+/- 2.81%)
  f1_score: 60.5

# 11. Testing Script & Metrics Visualization
Lastly, in the final parts we'll measure our model performance based on the following metrics:
- Accuracy: Calculate the ratio of correctly predicted labels
- Precision: Calculate how many positive predictions are truly positive (Minimize false positives)
- Recall: Calculate how many postivive labels are predicted (Minimize false negatives)
- F1-Score: Determine the mean of precision and recall combined
- ROC-AUC Score: Determine how confident our model is at classifying the classes (0.5 - Random Guessing, ~1.0: Almost 100% accurate)
- Confusion Matrix: Determine the total True Positive, False Positives, True Negatives and False Negatives that our model made

In [11]:
def evaluate_and_log(model, test_loader, device, tracker):
    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(pixel_values=inputs)

            probs = F.softmax(outputs.logits, dim=1)
            _, predicted = torch.max(outputs.logits, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='macro')
    cm = confusion_matrix(all_labels, all_preds)

    try:
        roc_auc = roc_auc_score(
            np.array(all_labels),
            np.array(all_probs),
            multi_class='ovr'
        )
    except ValueError:
        roc_auc = 0.0

    tracker.log_final_metrics("test", acc, prec, rec, f1, roc_auc, cm)
    tracker.save_all()

    print(f"\nTEST RESULTS (Best model from Fold {best_fold})")
    print(f"Accuracy:  {acc * 100:.2f}%")
    print(f"Precision: {prec * 100:.2f}%")
    print(f"Recall:    {rec * 100:.2f}%")
    print(f"F1 Score:  {f1 * 100:.2f}%")
    print(f"ROC-AUC:   {roc_auc * 100:.2f}%")
    print(f"Confusion Matrix:\n{cm}")

    return acc, prec, rec, f1, roc_auc, cm

# Load best model from K-Fold training and evaluate on held-out test set
best_model = Swinv2ForImageClassification.from_pretrained(
    'microsoft/swinv2-base-patch4-window8-256',
    num_labels=5,
    ignore_mismatched_sizes=True
).to(device)

best_model.load_state_dict(torch.load(os.path.join(tracker.exp_dir, "best_model.pth"), map_location=device))

evaluate_and_log(best_model, test_loader, device, tracker)

You passed `num_labels=5` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Swinv2ForImageClassification LOAD REPORT from: microsoft/swinv2-base-patch4-window8-256
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([5])            
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 1024]) vs model:torch.Size([5, 1024])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.



TEST RESULTS (Best model from Fold 4)
Accuracy:  62.96%
Precision: 61.17%
Recall:    65.57%
F1 Score:  55.13%
ROC-AUC:   91.93%
Confusion Matrix:
[[348 100   0   0   0]
 [  2  87   9   5   2]
 [  0  79  91  91  16]
 [  0   2   5  36   1]
 [  0   9  13  32  60]]


(0.6295546558704453,
 0.6117114886708759,
 0.6556749212216626,
 0.5512705662365185,
 0.9193213129624782,
 array([[348, 100,   0,   0,   0],
        [  2,  87,   9,   5,   2],
        [  0,  79,  91,  91,  16],
        [  0,   2,   5,  36,   1],
        [  0,   9,  13,  32,  60]]))